In [14]:
from graphdatascience import GraphDataScience
gds = GraphDataScience("bolt://localhost:7687", auth=("neo4j", "Lastfm_2026_pw"))
print(gds.server_version())

2.13.15


""


In [ ]:
gds.run_cypher("MATCH (n) DETACH DELETE n ")

In [13]:
print(gds.run_cypher("MATCH (n) RETURN count(n) AS nodes"))

   nodes
0      0


In [16]:
import pandas as pd, importlib.resources as r, graphdatascience, os

base = os.path.join(os.path.dirname(graphdatascience.__file__), "resources", "lastfm")
users  = pd.read_parquet(f"{base}/user_nodes.parquet.gzip")
arts   = pd.read_parquet(f"{base}/artist_nodes.parquet.gzip")
friends= pd.read_parquet(f"{base}/user_friend_df_directed.parquet.gzip")
listen = pd.read_parquet(f"{base}/user_listen_artist_rels.parquet.gzip")
tagged = pd.read_parquet(f"{base}/user_tag_artist_rels.parquet.gzip")

def run(q, rows, size=5000):
    for i in range(0, len(rows), size):
        gds.run_cypher(q, {"rows": rows[i:i+size]})

gds.run_cypher("CREATE CONSTRAINT user_id IF NOT EXISTS FOR (u:User) REQUIRE u.id IS UNIQUE")
gds.run_cypher("CREATE CONSTRAINT artist_id IF NOT EXISTS FOR (a:Artist) REQUIRE a.id IS UNIQUE")

run("UNWIND $rows AS r MERGE (:User {id:r.nodeId}) ", users[["nodeId"]].to_dict("records"))
run("UNWIND $rows AS r MERGE (:Artist {id:r.nodeId, rawId:r.rawId})", arts[["nodeId","rawId"]].to_dict("records"))
run("UNWIND $rows AS r MATCH (a:User {id:r.sourceNodeId}),(b:User {id:r.targetNodeId}) CREATE (a)-[:IS_FRIEND]->(b)",
    friends[["sourceNodeId","targetNodeId"]].to_dict("records"))
run("UNWIND $rows AS r MATCH (a:User {id:r.sourceNodeId}),(b:Artist {id:r.targetNodeId}) CREATE (a)-[:LISTEN_TO {weight:r.weight}]->(b)",
    listen[["sourceNodeId","targetNodeId","weight"]].to_dict("records"))
run("UNWIND $rows AS r MATCH (a:User {id:r.sourceNodeId}),(b:Artist {id:r.targetNodeId}) CREATE (a)-[:TAGGED {tagID:r.tagID, timestamp:r.timestamp}]->(b)",
    tagged[["sourceNodeId","targetNodeId","tagID","timestamp"]].to_dict("records"))

print(gds.run_cypher("MATCH (n) RETURN labels(n) AS label, count(*) AS n"))
print(gds.run_cypher("MATCH ()-[r]->() RETURN type(r) AS rel, count(*) AS n"))

      label      n
0    [User]   1892
1  [Artist]  18022
         rel       n
0  IS_FRIEND   12717
1  LISTEN_TO   92834
2     TAGGED  186479


In [20]:
gds.run_cypher("CALL gds.graph.drop('lastfm', false)")
print(gds.run_cypher("CALL gds.graph.list() YIELD graphName RETURN graphName"))

Received notification from DBMS server: <GqlStatusObject gql_status='01N03', status_description='warn: procedure field deprecated. The field `schema` of procedure gds.graph.drop() is deprecated.', position=<SummaryInputPosition line=1, column=1, offset=0>, raw_classification='DEPRECATION', classification=<NotificationClassification.DEPRECATION: 'DEPRECATION'>, raw_severity='WARNING', severity=<NotificationSeverity.WARNING: 'WARNING'>, diagnostic_record={'_classification': 'DEPRECATION', '_severity': 'WARNING', '_position': {'offset': 0, 'line': 1, 'column': 1}, 'OPERATION': '', 'OPERATION_CODE': '0', 'CURRENT_SCHEMA': '/'}> for query: "CALL gds.graph.drop('lastfm', false)"


Empty DataFrame
Columns: [graphName]
Index: []


In [21]:
print(gds.run_cypher("MATCH (n) RETURN labels(n) AS label, count(*) AS n"))
print(gds.run_cypher("MATCH ()-[r]->() RETURN type(r) AS rel, count(*) AS n"))

      label      n
0    [User]   1892
1  [Artist]  18022
         rel       n
0  IS_FRIEND   12717
1  LISTEN_TO   92834
2     TAGGED  186479


In [22]:
G, res = gds.graph.project.native(
    "listen",
    ["User", "Artist"],
    {"LISTEN_TO": {"orientation": "UNDIRECTED", "properties": "weight"}},
    overwrite=True,
)
print(G.node_count(), G.relationship_count())

 Loading:   0%|          | 0.0/100 [00:00<?, ?%/s]

19914 185668


In [26]:
checks = {
 "1 self-loops": "MATCH (a)-[r]->(a) RETURN type(r) AS rel, count(r) AS selfLoops",
 "2 duplicate pairs": """MATCH (a)-[r:IS_FRIEND|LISTEN_TO]->(b)
     WITH a, b, type(r) AS rel, count(*) AS c WHERE c > 1
     RETURN rel, count(*) AS duplicatedPairs""",
 "3 tagged rows vs pairs": """MATCH (u:User)-[r:TAGGED]->(a:Artist)
     RETURN count(r) AS taggedRows, count(DISTINCT [u.id, a.id]) AS distinctPairs""",
 "4 reciprocal friends": """MATCH (a:User)-[:IS_FRIEND]->(b:User)
     RETURN count(*) AS directed,
            sum(CASE WHEN (b)-[:IS_FRIEND]->(a) THEN 1 ELSE 0 END) AS reciprocal""",
 "5 isolated nodes": "MATCH (n) WHERE NOT (n)--() RETURN labels(n) AS label, count(*) AS isolated",
 "6a artists never listened": "MATCH (a:Artist) WHERE NOT ()-[:LISTEN_TO]->(a) RETURN count(a) AS artistsNoListen",
 "6b users never listened": "MATCH (u:User) WHERE NOT (u)-[:LISTEN_TO]->() RETURN count(u) AS usersNoListen",
}
for name, q in checks.items():
    print("==", name); print(gds.run_cypher(q).to_string(index=False)); print()

== 1 self-loops
Empty DataFrame
Columns: [rel, selfLoops]
Index: []

== 2 duplicate pairs
Empty DataFrame
Columns: [rel, duplicatedPairs]
Index: []

== 3 tagged rows vs pairs
 taggedRows  distinctPairs
     186479          71064

== 4 reciprocal friends
 directed  reciprocal
    12717           0

== 5 isolated nodes
Empty DataFrame
Columns: [label, isolated]
Index: []

== 6a artists never listened
 artistsNoListen
             390

== 6b users never listened
 usersNoListen
             0



In [24]:
print(gds.run_cypher("MATCH (a:Artist) WHERE NOT ()-[:LISTEN_TO]->(a) RETURN count(a) AS artistsNoListen"))

   artistsNoListen
0              390
